# Notebook 6 — Stage 4: Evaluation
**MAI 656 — Natural Language Processing | Canadian University Dubai | Spring 2026**

This notebook evaluates the fine-tuned Qwen2.5-VL model on the held-out eval set, reporting:
- **CER** — Character Error Rate
- **WER** — Word Error Rate
- **Exact match rate**
- **Valid JSON rate**
- **Average inference time per sample**

> ⚠️ **A GPU with ≥24 GB VRAM is required** (same machine as training, or another GPU instance).

**Input:**
- `data/eval/eval.jsonl`
- `models/lora_adapter/` (saved by Notebook 5)

**Output:** `logs/eval_results.json`

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install transformers==4.47.0 peft==0.14.0 accelerate==1.2.1 bitsandbytes==0.45.0 --quiet
!{sys.executable} -m pip install qwen-vl-utils --quiet

## 3. Set Project Root

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/home/ubuntu/nlp_project')

assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'

print(f'Project root: {PROJECT_ROOT}')

## 4. Configuration

In [ ]:
from pathlib import Path

BASE_MODEL = 'Qwen/Qwen2.5-VL-7B-Instruct'
MIN_PIXELS = 4   * 28 * 28   # must match training
MAX_PIXELS = 128 * 28 * 28   # must match training

ADAPTER_DIR = Path(PROJECT_ROOT) / 'models' / 'lora_adapter'
EVAL_DATA   = Path(PROJECT_ROOT) / 'data'   / 'eval' / 'eval.jsonl'
LOG_DIR     = Path(PROJECT_ROOT) / 'logs'

LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Adapter: {ADAPTER_DIR}')
print(f'Eval:    {EVAL_DATA}')

## 5. Load Base Model + LoRA Adapter

In [ ]:
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import PeftModel

print('Loading base model...')
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='sdpa',
    local_files_only=True,
)

print('Loading LoRA adapter...')
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()

processor = AutoProcessor.from_pretrained(
    BASE_MODEL,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=True,
)

print('Model ready.')

In [ ]:
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import PeftModel

print('Loading base model...')
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='sdpa',
)

print('Loading LoRA adapter...')
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()

processor = AutoProcessor.from_pretrained(
    BASE_MODEL,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

print('Model ready.')

## 6. Define Metrics and JSON Parser

In [ ]:
import json
import difflib

def parse_output(raw_text: str) -> dict:
    """Try multiple strategies to extract valid JSON."""
    text = raw_text.strip()

    # Strategy 1: Direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Strategy 2: Strip markdown code blocks
    if '```' in text:
        text = text.split('```json')[-1].split('```')[0].strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass

    # Strategy 3: Find last closing brace
    for i in range(len(text) - 1, -1, -1):
        if text[i] == '}':
            try:
                return json.loads(text[:i+1])
            except json.JSONDecodeError:
                continue

    # Strategy 4: Auto-complete common truncations
    for suffix in ['"}',' "}', '"}', '"]}', '"}}']:
        try:
            return json.loads(text + suffix)
        except json.JSONDecodeError:
            continue

    return None


def char_error_rate(prediction: str, reference: str) -> float:
    if not reference:
        return 0.0 if not prediction else 1.0
    matcher = difflib.SequenceMatcher(None, reference, prediction)
    distance = sum(
        max(j2 - j1, i2 - i1)
        for tag, i1, i2, j1, j2 in matcher.get_opcodes()
        if tag != 'equal'
    )
    return distance / max(len(reference), 1)


def word_error_rate(prediction: str, reference: str) -> float:
    ref_words  = reference.split()
    pred_words = prediction.split()
    if not ref_words:
        return 0.0 if not pred_words else 1.0
    matcher = difflib.SequenceMatcher(None, ref_words, pred_words)
    distance = sum(
        max(j2 - j1, i2 - i1)
        for tag, i1, i2, j1, j2 in matcher.get_opcodes()
        if tag != 'equal'
    )
    return distance / max(len(ref_words), 1)

print('Metric functions defined.')

## 7. Run Evaluation

In [ ]:
import time

# Load eval samples
eval_samples = []
with open(EVAL_DATA) as f:
    for line in f:
        eval_samples.append(json.loads(line))
print(f'Evaluating {len(eval_samples)} samples...')

# Build token suppression list (prevents <tool_call> hallucinations)
suppress_tokens = ['<tool_call>', '<|tool_call|>', '<tool_response>', '<|tool_response|>']
bad_words_ids = []
for token in suppress_tokens:
    ids = processor.tokenizer.encode(token, add_special_tokens=False)
    if ids:
        bad_words_ids.append(ids)

results = []
for i, sample in enumerate(eval_samples):
    messages = sample['messages']
    expected = messages[-1]['content']      # ground truth
    input_messages = messages[:-1]          # system + user

    text = processor.apply_chat_template(
        input_messages, tokenize=False, add_generation_prompt=True
    )

    # JSON prefix forcing — steers the model to produce valid JSON immediately
    json_prefix = '{"transcription":"'
    text += json_prefix

    inputs = processor(text=text, return_tensors='pt').to(model.device)

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False,
            bad_words_ids=bad_words_ids if bad_words_ids else None,
        )
    inference_time = time.time() - start_time

    generated = processor.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    generated = json_prefix + generated  # re-attach prefix

    pred_parsed = parse_output(generated)
    ref_parsed  = parse_output(expected)

    pred_text = pred_parsed.get('transcription', '') if pred_parsed else ''
    ref_text  = ref_parsed.get('transcription',  '') if ref_parsed  else ''

    cer   = char_error_rate(pred_text, ref_text)
    wer   = word_error_rate(pred_text, ref_text)
    exact = pred_text.strip() == ref_text.strip()

    results.append({
        'cer': cer, 'wer': wer,
        'exact_match': exact,
        'valid_json': pred_parsed is not None,
        'inference_time': inference_time,
        'prediction': pred_text,
        'reference': ref_text,
    })

    print(f'[{i+1}/{len(eval_samples)}] CER={cer:.3f}  WER={wer:.3f}  Time={inference_time:.1f}s')

print('\nEvaluation complete.')

## 8. Summary & Save Results

In [ ]:
n = len(results)

avg_cer    = sum(r['cer']            for r in results) / n
avg_wer    = sum(r['wer']            for r in results) / n
exact_rate = sum(r['exact_match']    for r in results) / n * 100
json_rate  = sum(r['valid_json']     for r in results) / n * 100
avg_time   = sum(r['inference_time'] for r in results) / n

summary = {
    'num_samples':        n,
    'avg_cer':            round(avg_cer,    4),
    'avg_wer':            round(avg_wer,    4),
    'exact_match_rate':   round(exact_rate, 2),
    'valid_json_rate':    round(json_rate,  2),
    'avg_inference_time': round(avg_time,   2),
}

print('=== Evaluation Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

output_path = LOG_DIR / 'eval_results.json'
with open(output_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\nResults saved to {output_path}')

## 9. Inspect Worst Predictions

In [ ]:
# Show the 5 samples with the highest CER
worst = sorted(results, key=lambda r: r['cer'], reverse=True)[:5]

print('=== Top 5 Highest CER Samples ===')
for i, r in enumerate(worst):
    print(f'\n#{i+1}  CER={r["cer"]:.3f}  WER={r["wer"]:.3f}')
    print(f'  Reference:  {r["reference"]}')
    print(f'  Prediction: {r["prediction"]}')

## Next Step (Optional)

If you want to serve the model as an API endpoint, continue to:

**Notebook 7 → `07_inference_server.ipynb`** — Launch a vLLM inference server